In [ ]:
import numpy as np
import open3d as o3d
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN

In [ ]:
def remove_outliers(point_cloud, nb_neighbors, std_ratio, nb_points, radius):
    # Remoção de outliers estatísticos
    filtered_stat, ind_stat = point_cloud.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    print(f"Pontos após remoção estatística de outliers: {len(filtered_stat.points)}")

    # Remoção de outliers por raio
    filtered_radius, ind_radius = filtered_stat.remove_radius_outlier(nb_points=nb_points, radius=radius)
    print(f"Pontos após remoção por raio de outliers: {len(filtered_radius.points)}")

    return filtered_radius

def dbscan_clustering(point_cloud, eps, min_samples):
    points = np.asarray(point_cloud.points)
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    labels = clustering.labels_

    # Identificar o maior cluster (ignorar outliers com label -1)
    unique_labels, counts = np.unique(labels[labels != -1], return_counts=True)
    if len(unique_labels) == 0:
        print("Nenhum cluster encontrado")
        return point_cloud
    largest_cluster_label = unique_labels[np.argmax(counts)]

    # Filtrar pontos do maior cluster
    largest_cluster_points = points[labels == largest_cluster_label]
    filtered_pc = o3d.geometry.PointCloud()
    filtered_pc.points = o3d.utility.Vector3dVector(largest_cluster_points)
    print(f"Pontos após DBSCAN: {len(filtered_pc.points)}")

    return filtered_pc

def visualize_point_clouds(original_pc, filtered_pc, clustered_pc):
    x1 = [p[0] for p in original_pc.points]
    y1 = [p[1] for p in original_pc.points]
    z1 = [p[2] for p in original_pc.points]

    x2 = [p[0] for p in filtered_pc.points]
    y2 = [p[1] for p in filtered_pc.points]
    z2 = [p[2] for p in filtered_pc.points]

    x3 = [p[0] for p in clustered_pc.points]
    y3 = [p[1] for p in clustered_pc.points]
    z3 = [p[2] for p in clustered_pc.points]

    trace1 = go.Scatter3d(
        x=x1, y=y1, z=z1,
        mode='markers',
        marker=dict(
            size=3,
            color='blue',
        ),
        name='Original'
    )

    trace2 = go.Scatter3d(
        x=x2, y=y2, z=z2,
        mode='markers',
        marker=dict(
            size=3,
            color='red',
        ),
        name='Filtered'
    )

    trace3 = go.Scatter3d(
        x=x3, y=y3, z=z3,
        mode='markers',
        marker=dict(
            size=3,
            color='green',
        ),
        name='Clustered'
    )
    trace4 = go.Scatter3d(
    x=x4, y=y4, z=z4,
    mode='markers',
    marker=dict(
        size=3,
        color='gray',
    ),
    name='Truck bucket'
)

    fig = go.Figure()
    fig.add_trace(trace1)
    fig.add_trace(trace2)
    fig.add_trace(trace3)
    fig.add_trace(trace4)

    # Atualizar layout
    fig.update_layout(
        legend=dict(
            x=0.8,
            y=0.8,
            traceorder="normal",
            font=dict(
                family="sans-serif",
                size=12,
                color="black"
            ),
            bgcolor="LightSteelBlue",
            bordercolor="Black",
            borderwidth=2
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        scene=dict(
            xaxis=dict(title='X'),
            yaxis=dict(title='Y'),
            zaxis=dict(title='Z')
        )
    )

    # Mostrar figura
    fig.show()

In [ ]:
# Carregar nuvem de pontos
aligned = o3d.io.read_point_cloud('caminho_para_sua_nuvem_de_pontos.pcd')

# Parâmetros de filtragem conforme seu código
nb_neighbors = 40
std_ratio = 1.0
nb_points = 50
radius = 250.0

# Parâmetros do DBSCAN
eps = 49.619  # Ajuste este valor conforme necessário
min_samples = 7  # Ajuste este valor conforme necessário

# Remover outliers
filtered_pc = remove_outliers(aligned, nb_neighbors, std_ratio, nb_points, radius)

# Aplicar DBSCAN para remover outliers adicionais
clustered_pc = dbscan_clustering(filtered_pc, eps, min_samples)

# Visualizar nuvens de pontos
visualize_point_clouds(aligned, filtered_pc, clustered_pc)